# Chapter 18 — Product Measures and the Fubini–Tonelli Theorem

Companion notebook to Chapter 18 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Integrals are computed exactly with fractions, or by the midpoint rule on a
fine grid, which is deterministic. Random samples are used only in the
sections on Monte Carlo, where a cell says so.

In [1]:
import math
import warnings
from fractions import Fraction
from itertools import product

import torch

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor


def midpoints(a, b, n):
    # Midpoints of n equal cells of [a, b] and the cell length.
    h = (b - a) / n
    return a + h * (torch.arange(n) + 0.5), h


def integral(f, a, b, n=2_000_000):
    x, h = midpoints(a, b, n)
    return (f(x) * h).sum()

2.15.0+cu132


## 1. A product measure on a finite set (Theorem 18.5)

On finite sets every subset is measurable and a measure is a list of
weights. The product measure gives the weight $\mu\{x\}\nu\{y\}$ to the
pair $(x, y)$. The measure of a set $E$ is computed in three ways: point by
point, by integrating $\nu(E_x)$, and by integrating $\mu(E^y)$.

In [2]:
mu = {0: Fraction(1, 2), 1: Fraction(1, 3), 2: Fraction(1, 6)}       # on X
nu = {"a": Fraction(2), "b": Fraction(1, 4), "c": Fraction(3, 4)}    # on Y, not a probability

E = {(0, "a"), (0, "c"), (1, "b"), (2, "a"), (2, "b"), (2, "c")}

direct = sum(mu[x] * nu[y] for x, y in E)
by_x = sum(mu[x] * sum(nu[y] for y in nu if (x, y) in E) for x in mu)   # int nu(E_x) dmu
by_y = sum(nu[y] * sum(mu[x] for x in mu if (x, y) in E) for y in nu)   # int mu(E^y) dnu
print("sections E_x:", {x: sorted(y for y in nu if (x, y) in E) for x in mu})
print(f"point by point = {direct},  slices in x = {by_x},  slices in y = {by_y}")
assert direct == by_x == by_y

# On rectangles the measure is a product.
for A in [{0}, {0, 2}, {0, 1, 2}]:
    for B in [{"a"}, {"b", "c"}]:
        rect = sum(mu[x] * nu[y] for x in A for y in B)
        assert rect == sum(mu[x] for x in A) * sum(nu[y] for y in B)
print("rectangles: (mu x nu)(A x B) = mu(A) nu(B)")

sections E_x: {0: ['a', 'c'], 1: ['b'], 2: ['a', 'b', 'c']}
point by point = 47/24,  slices in x = 47/24,  slices in y = 47/24
rectangles: (mu x nu)(A x B) = mu(A) nu(B)


## 2. A set is measured by its slices (Figure 18.1, Proposition 18.7)

The unit disc sliced vertically: its section at $x$ has length
$2\sqrt{1 - x^2}$, and the integral of the lengths is $\pi$.

In [3]:
area = integral(lambda x: 2 * torch.sqrt(1 - x**2), -1.0, 1.0)
print(f"integral of the lengths of the sections = {area:.8f}   pi = {math.pi:.8f}")
assert abs(area - math.pi) < 1e-7

integral of the lengths of the sections = 3.14159265   pi = 3.14159265


The region under a graph, sliced both ways:
$\int f\,d\mu = \int_0^\infty\mu\{f > t\}\,dt$. First for Lebesgue measure on
$[0, 5]$ and the function of Figure 18.1.

In [4]:
def f_fig(x):
    return 0.45 + 2.3 * torch.sin(torch.deg2rad(72 * x))**2 * torch.exp(-0.22 * x)


x, hx = midpoints(0.0, 5.0, 200_000)
fx = f_fig(x)
vertical = (fx * hx).sum()                               # int f dx

t, ht = midpoints(0.0, fx.max().item(), 4000)
# mu{f > t}: the length of the set where f exceeds t, for each level t
fx_sorted = torch.sort(fx).values
count_above = len(fx) - torch.searchsorted(fx_sorted, t, right=True)
horizontal = (count_above * hx * ht).sum()               # int mu{f > t} dt

print(f"vertical slices   int f dx          = {vertical:.6f}")
print(f"horizontal slices int mu(f > t) dt  = {horizontal:.6f}")
assert abs(vertical - horizontal) < 2e-3

vertical slices   int f dx          = 5.710747
horizontal slices int mu(f > t) dt  = 5.710744


Then for a probability: $\mathbb E[Z] = \int_0^\infty P(Z > t)\,dt$. For the
empirical measure of a sample the right-hand side is exact: $P(Z > t)$ is a
step function. Exercise 18.5 is the same with $pt^{p-1}$ inside.

In [5]:
sample = [Fraction(3, 10), Fraction(7, 5), Fraction(7, 5), Fraction(2), Fraction(9, 2)]
N = len(sample)
mean = sum(sample) / N

levels = [Fraction(0)] + sorted(set(sample))
tail = sum((b - a) * Fraction(sum(1 for z in sample if z > a), N)
           for a, b in zip(levels[:-1], levels[1:]))        # int P(Z > t) dt
print(f"mean = {mean},  integral of the tail = {tail}")
assert mean == tail

# Exercise 18.5 (2): uniform on [0, 1] and the exponential law, p = 1, 2, 3.5
for p in (1.0, 2.0, 3.5):
    I = integral(lambda t: p * t**(p - 1) * (1 - t), 0.0, 1.0).item()
    print(f"uniform, p = {p}:  int p t^(p-1) P(Z > t) dt = {I:.8f}   1/(p+1) = {1 / (p + 1):.8f}")
    assert abs(I - 1 / (p + 1)) < 1e-8
I = integral(lambda t: 2 * t * torch.exp(-t), 0.0, 60.0).item()
print(f"exponential, p = 2: {I:.8f}   E[Z^2] = 2")
assert abs(I - 2.0) < 1e-8

mean = 48/25,  integral of the tail = 48/25
uniform, p = 1.0:  int p t^(p-1) P(Z > t) dt = 0.50000000   1/(p+1) = 0.50000000
uniform, p = 2.0:  int p t^(p-1) P(Z > t) dt = 0.33333333   1/(p+1) = 0.33333333
uniform, p = 3.5:  int p t^(p-1) P(Z > t) dt = 0.22222222   1/(p+1) = 0.22222222
exponential, p = 2: 2.00000000   E[Z^2] = 2


## 3. Tonelli: exchanging non-negative sums and integrals

**Exercise 18.2.** $\sum_{n\geq2}(\zeta(n) - 1) = 1$: the double series
$\sum_{n\geq2}\sum_{k\geq2}k^{-n}$ summed in both orders.

In [6]:
K = 200_000
k = torch.arange(2, K + 1, dtype=torch.float64)
n = torch.arange(2, 61, dtype=torch.float64)

rows_first = sum((k**-nn).sum().item() for nn in n)        # sum over n of (zeta(n) - 1)
cols_first = (k**-2 / (1 - 1 / k)).sum().item()            # sum over k of the geometric series
print(f"sum over n first of (zeta(n) - 1) = {rows_first:.6f}")
print(f"sum over k first of 1/(k(k-1))    = {cols_first:.6f}   (exactly 1 - 1/K = {1 - 1 / K:.6f})")
assert abs(rows_first - 1) < 1e-5 and abs(cols_first - (1 - 1 / K)) < 1e-10

sum over n first of (zeta(n) - 1) = 0.999995
sum over k first of 1/(k(k-1))    = 0.999995   (exactly 1 - 1/K = 0.999995)


**Exercise 18.1.** $\int_0^1\int_y^1 e^{x^2}dx\,dy = (e - 1)/2$. The grid
computes the double integral over the triangle directly; the exchanged
order gives the closed form.

In [7]:
n = 3000
g, h = midpoints(0.0, 1.0, n)
X, Y = torch.meshgrid(g, g, indexing="ij")
inside = (Y <= X).double()                                  # the triangle 0 <= y <= x <= 1
double = (torch.exp(X**2) * inside).sum() * h * h
exchanged = integral(lambda x: x * torch.exp(x**2), 0.0, 1.0)
print(f"double integral on the triangle = {double:.5f}")
print(f"int x e^(x^2) dx                = {exchanged:.8f}   (e - 1)/2 = {(math.e - 1) / 2:.8f}")
assert abs(exchanged - (math.e - 1) / 2) < 1e-9 and abs(double - exchanged) < 2e-3

double integral on the triangle = 0.85938
int x e^(x^2) dx                = 0.85914091   (e - 1)/2 = 0.85914091


**Exercise 18.4.**
$\int_0^\infty\frac{e^{-ax} - e^{-bx}}{x}dx = \ln\frac ba$.

In [8]:
for a, b in [(1.0, 2.0), (0.5, 3.0), (2.0, 7.0)]:
    I = integral(lambda x: (torch.exp(-a * x) - torch.exp(-b * x)) / x, 0.0, 120.0, 6_000_000).item()
    print(f"a = {a}, b = {b}:  integral = {I:.8f}   ln(b/a) = {math.log(b / a):.8f}")
    assert abs(I - math.log(b / a)) < 1e-7

a = 1.0, b = 2.0:  integral = 0.69314718   ln(b/a) = 0.69314718


a = 0.5, b = 3.0:  integral = 1.79175947   ln(b/a) = 1.79175947


a = 2.0, b = 7.0:  integral = 1.25276297   ln(b/a) = 1.25276297


**Example 18.10, the Gaussian integral.**
$I^2 = \int_0^\infty\frac{ds}{2(1 + s^2)} = \frac\pi4$. The inner integral
$\int_0^\infty xe^{-x^2(1 + s^2)}dx = \frac1{2(1 + s^2)}$ is checked for a
few values of $s$, then $I$ itself.

In [9]:
for s in (0.0, 0.5, 2.0):
    inner = integral(lambda x: x * torch.exp(-x**2 * (1 + s**2)), 0.0, 12.0).item()
    assert abs(inner - 1 / (2 * (1 + s**2))) < 1e-9
I = integral(lambda x: torch.exp(-x**2), 0.0, 12.0).item()
print(f"I = {I:.10f}   sqrt(pi)/2 = {math.sqrt(math.pi) / 2:.10f}   I^2 = {I**2:.10f}   pi/4 = {math.pi / 4:.10f}")
assert abs(I**2 - math.pi / 4) < 1e-10
total = integral(lambda x: torch.exp(-x**2 / 2) / math.sqrt(2 * math.pi), -12.0, 12.0).item()
print(f"integral of the Gaussian density = {total:.10f}")
assert abs(total - 1) < 1e-10

I = 0.8862269255   sqrt(pi)/2 = 0.8862269255   I^2 = 0.7853981634   pi/4 = 0.7853981634
integral of the Gaussian density = 1.0000000000


## 4. When the exchange fails (pitfall of §18.3, Figure 18.2, Exercise 18.3)

The array $a_{mn}$: $+1$ on the diagonal, $-1$ just right of it. Every row
sums to $0$; the first column sums to $1$ and the others to $0$. A finite
truncation shows where the discrepancy sits: an $N\times N$ square has total
$1$, an $N\times(N+1)$ rectangle has total $0$.

In [10]:
def a(m, n):
    return 1 if n == m else (-1 if n == m + 1 else 0)


N = 60
row_sums = [sum(a(m, n) for n in range(1, m + 3)) for m in range(1, N + 1)]       # complete rows
col_sums = [sum(a(m, n) for m in range(1, n + 2)) for n in range(1, N + 1)]       # complete columns
print("row sums:   ", row_sums[:8], "...")
print("column sums:", col_sums[:8], "...")
assert all(r == 0 for r in row_sums) and col_sums[0] == 1 and all(c == 0 for c in col_sums[1:])

square = sum(a(m, n) for m in range(1, N + 1) for n in range(1, N + 1))
rectangle = sum(a(m, n) for m in range(1, N + 1) for n in range(1, N + 2))
absolute = sum(abs(a(m, n)) for m in range(1, N + 1) for n in range(1, N + 1))
print(f"N x N total = {square},  N x (N+1) total = {rectangle},  sum of |a| on N x N = {absolute}")
assert square == 1 and rectangle == 0 and absolute == 2 * N - 1

row sums:    [0, 0, 0, 0, 0, 0, 0, 0] ...
column sums: [1, 0, 0, 0, 0, 0, 0, 0] ...
N x N total = 1,  N x (N+1) total = 0,  sum of |a| on N x N = 119


The function $f(x, y) = \frac{x^2 - y^2}{(x^2 + y^2)^2}$ on $(0, 1)^2$. The
inner integrals have closed forms $\pm\frac1{1 + u^2}$ (checked at a few
points); the iterated integrals are $\pm\frac\pi4$. The integral of $|f|$
over $[\delta, 1]^2$ grows like $\ln(1/\delta)$.

In [11]:
f = lambda x, y: (x**2 - y**2) / (x**2 + y**2)**2

for u in (0.2, 0.5, 0.9):
    in_y = integral(lambda y: f(T(u), y), 0.0, 1.0).item()        # int f(u, y) dy
    in_x = integral(lambda x: f(x, T(u)), 0.0, 1.0).item()        # int f(x, u) dx
    assert abs(in_y - 1 / (1 + u**2)) < 1e-6 and abs(in_x + 1 / (1 + u**2)) < 1e-6

dy_then_dx = integral(lambda x: 1 / (1 + x**2), 0.0, 1.0).item()
dx_then_dy = integral(lambda y: -1 / (1 + y**2), 0.0, 1.0).item()
print(f"dy then dx: {dy_then_dx:+.8f}    dx then dy: {dx_then_dy:+.8f}    pi/4 = {math.pi / 4:.8f}")
assert abs(dy_then_dx - math.pi / 4) < 1e-10 and abs(dx_then_dy + math.pi / 4) < 1e-10

# int |f| over [delta, 1]^2 = 2 * int_delta^1 ( int_delta^x f dy ) dx, with the
# inner integral in closed form: x/(2 x^2) - delta/(x^2 + delta^2).
previous = -math.inf
for delta in (1e-1, 1e-2, 1e-3, 1e-4):
    I = 2 * integral(lambda x: 1 / (2 * x) - delta / (x**2 + delta**2), delta, 1.0, 4_000_000).item()
    print(f"delta = {delta:.0e}:  integral of |f| on [delta, 1]^2 = {I:.4f}   ln(1/delta) = {math.log(1 / delta):.4f}")
    assert I > previous + 2
    previous = I

dy then dx: +0.78539816    dx then dy: -0.78539816    pi/4 = 0.78539816


delta = 1e-01:  integral of |f| on [delta, 1]^2 = 0.9311   ln(1/delta) = 2.3026
delta = 1e-02:  integral of |f| on [delta, 1]^2 = 3.0544   ln(1/delta) = 4.6052


delta = 1e-03:  integral of |f| on [delta, 1]^2 = 5.3390   ln(1/delta) = 6.9078
delta = 1e-04:  integral of |f| on [delta, 1]^2 = 7.6397   ln(1/delta) = 9.2103


## 5. The simplex (Exercise 18.6)

$\lambda^n(\Delta_n) = 1/n!$, by the recursion
$v_n = v_{n-1}\int_0^1(1 - t)^{n-1}dt$, and by sampling the unit cube (this
cell uses a seeded random generator).

In [12]:
v = Fraction(1)
for n in range(2, 8):
    v = v * Fraction(1, n)             # int_0^1 (1 - t)^(n-1) dt = 1/n
    assert v == Fraction(1, math.factorial(n))
print("v_n = 1/n! for n = 1..7 by the recursion")

M = 4_000_000
for n in (2, 3, 4, 5):
    U = torch.rand(M, n)
    frac = (U.sum(dim=1) <= 1).double().mean().item()
    exact = 1 / math.factorial(n)
    se = math.sqrt(exact * (1 - exact) / M)
    print(f"n = {n}:  fraction of the cube = {frac:.6f}   1/n! = {exact:.6f}   (standard error {se:.6f})")
    assert abs(frac - exact) < 5 * se

v_n = 1/n! for n = 1..7 by the recursion


n = 2:  fraction of the cube = 0.500169   1/n! = 0.500000   (standard error 0.000250)


n = 3:  fraction of the cube = 0.166787   1/n! = 0.166667   (standard error 0.000186)


n = 4:  fraction of the cube = 0.041569   1/n! = 0.041667   (standard error 0.000100)


n = 5:  fraction of the cube = 0.008400   1/n! = 0.008333   (standard error 0.000045)


## 6. Independence (Definition 18.12, Proposition 18.13)

Two dice. Under the product law the expectation of a product is the product
of the expectations, for any $g$ and $h$. Under a law with the same
marginals that is not a product, it is not.

In [13]:
faces = range(1, 7)
uniform = {z: Fraction(1, 6) for z in faces}
joint_indep = {(a, b): uniform[a] * uniform[b] for a in faces for b in faces}
joint_equal = {(a, b): (Fraction(1, 6) if a == b else Fraction(0)) for a in faces for b in faces}

g = lambda a: a**2
h = lambda b: 1 if b % 2 == 0 else 0


def E(joint, fn):
    return sum(fn(a, b) * p for (a, b), p in joint.items())


for name, joint in [("independent", joint_indep), ("Z2 = Z1", joint_equal)]:
    # both laws have uniform marginals
    assert all(sum(joint[(a, b)] for b in faces) == Fraction(1, 6) for a in faces)
    assert all(sum(joint[(a, b)] for a in faces) == Fraction(1, 6) for b in faces)
    lhs = E(joint, lambda a, b: g(a) * h(b))
    rhs = E(joint, lambda a, b: g(a)) * E(joint, lambda a, b: h(b))
    var_sum = E(joint, lambda a, b: (a + b)**2) - E(joint, lambda a, b: a + b)**2
    print(f"{name:12s} E[g h] = {lhs}   E[g] E[h] = {rhs}   Var(Z1 + Z2) = {var_sum}")

var_one = E(joint_indep, lambda a, b: a**2) - E(joint_indep, lambda a, b: a)**2
assert E(joint_indep, lambda a, b: g(a) * h(b)) == Fraction(91, 6) * Fraction(1, 2)
assert E(joint_equal, lambda a, b: g(a) * h(b)) != Fraction(91, 6) * Fraction(1, 2)
assert E(joint_indep, lambda a, b: (a + b)**2) - E(joint_indep, lambda a, b: a + b)**2 == 2 * var_one
assert E(joint_equal, lambda a, b: (a + b)**2) - E(joint_equal, lambda a, b: a + b)**2 == 4 * var_one
print(f"Var(Z) = {var_one}: variances add for independent dice, and double again when Z2 = Z1")

independent  E[g h] = 91/12   E[g] E[h] = 91/12   Var(Z1 + Z2) = 35/6
Z2 = Z1      E[g h] = 28/3   E[g] E[h] = 91/12   Var(Z1 + Z2) = 35/3
Var(Z) = 35/12: variances add for independent dice, and double again when Z2 = Z1


## 7. Monte Carlo (Theorem 18.14)

For $N$ independent samples the average has variance $\sigma^2/N$. This is
checked exactly on a die, by enumerating all $6^N$ outcomes of the product
law.

In [14]:
m_exact = Fraction(7, 2)
sigma2 = Fraction(35, 12)
for N in (1, 2, 3, 4, 5):
    mean = Fraction(0)
    second = Fraction(0)
    for outcome in product(faces, repeat=N):          # the product measure on {1..6}^N
        avg = Fraction(sum(outcome), N)
        mean += avg
        second += avg**2
    mean /= 6**N
    var = second / 6**N - mean**2
    print(f"N = {N}:  E[average] = {mean}   Var(average) = {var} = sigma^2 / {sigma2 / var}")
    assert mean == m_exact and var == sigma2 / N

N = 1:  E[average] = 7/2   Var(average) = 35/12 = sigma^2 / 1
N = 2:  E[average] = 7/2   Var(average) = 35/24 = sigma^2 / 2
N = 3:  E[average] = 7/2   Var(average) = 35/36 = sigma^2 / 3
N = 4:  E[average] = 7/2   Var(average) = 35/48 = sigma^2 / 4
N = 5:  E[average] = 7/2   Var(average) = 7/12 = sigma^2 / 5


The same with random samples (seeded), in dimension $1$ and in dimension
$50$: $g(x) = \cos(x_1 + \cdots + x_d)$ for $x$ uniform in the cube
$[0, 1]^d$. The root-mean-square error times $\sqrt N$ stays near $\sigma$
in both, and the bound $P(|\hat g_N - m|\geq\epsilon)\leq\sigma^2/(N\epsilon^2)$
holds.

In [15]:
def exact_mean(d):
    # E[cos(sum x_i)] = Re( E[e^{i x}]^d ),  E[e^{i x}] = (e^i - 1) / i
    c = (complex(math.cos(1), math.sin(1)) - 1) / 1j
    return (c**d).real


def exact_var(d):
    # E[cos^2 S] = (1 + E[cos 2S]) / 2,  E[e^{2 i x}] = (e^{2i} - 1) / (2i)
    c2 = (complex(math.cos(2), math.sin(2)) - 1) / 2j
    return (1 + (c2**d).real) / 2 - exact_mean(d)**2


R = 2000                                   # independent repetitions of the estimator
for d in (1, 50):
    m, s2 = exact_mean(d), exact_var(d)
    print(f"d = {d}:  m = {m:+.6f}   sigma = {math.sqrt(s2):.4f}")
    for N in (100, 400, 1600):
        est = torch.cos(torch.rand(R, N, d).sum(dim=2)).mean(dim=1)
        rmse = ((est - m)**2).mean().sqrt().item()
        eps = 3 * math.sqrt(s2 / N)
        freq = ((est - m).abs() >= eps).double().mean().item()
        print(f"   N = {N:5d}:  rmse = {rmse:.5f}   rmse * sqrt(N) = {rmse * math.sqrt(N):.4f}"
              f"   P(|err| >= 3 sigma/sqrt N) = {freq:.4f} <= 1/9")
        assert abs(rmse * math.sqrt(N) - math.sqrt(s2)) < 0.08 * math.sqrt(s2)
        assert freq <= 1 / 9

d = 1:  m = +0.841471   sigma = 0.1387
   N =   100:  rmse = 0.01393   rmse * sqrt(N) = 0.1393   P(|err| >= 3 sigma/sqrt N) = 0.0010 <= 1/9
   N =   400:  rmse = 0.00712   rmse * sqrt(N) = 0.1423   P(|err| >= 3 sigma/sqrt N) = 0.0025 <= 1/9
   N =  1600:  rmse = 0.00348   rmse * sqrt(N) = 0.1391   P(|err| >= 3 sigma/sqrt N) = 0.0055 <= 1/9
d = 50:  m = +0.121261   sigma = 0.6967


   N =   100:  rmse = 0.06945   rmse * sqrt(N) = 0.6945   P(|err| >= 3 sigma/sqrt N) = 0.0025 <= 1/9


   N =   400:  rmse = 0.03533   rmse * sqrt(N) = 0.7065   P(|err| >= 3 sigma/sqrt N) = 0.0035 <= 1/9


   N =  1600:  rmse = 0.01746   rmse * sqrt(N) = 0.6985   P(|err| >= 3 sigma/sqrt N) = 0.0020 <= 1/9


**Exercise 18.8, the square of an expectation.** With $g(Z)\in\{0, 1\}$
with probability $\frac12$ and $N = 2$: the square of the average has mean
$\frac38$, the product of two independent samples has mean
$\frac14 = m^2$.

In [16]:
coin = {0: Fraction(1, 2), 1: Fraction(1, 2)}
square_of_average = sum(Fraction(a + b, 2)**2 * coin[a] * coin[b] for a in coin for b in coin)
product_of_two = sum(a * b * coin[a] * coin[b] for a in coin for b in coin)
print(f"E[average^2] = {square_of_average},  E[g(Z1) g(Z2)] = {product_of_two},  m^2 = 1/4")
assert square_of_average == Fraction(3, 8) == Fraction(1, 4) + Fraction(1, 4) / 2
assert product_of_two == Fraction(1, 4)

# General N: bias sigma^2 / N, exactly.
for N in (2, 3, 6):
    val = sum(Fraction(sum(o), N)**2 for o in product((0, 1), repeat=N)) / 2**N
    assert val == Fraction(1, 4) + Fraction(1, 4) / N
print("E[average^2] = m^2 + sigma^2 / N for N = 2, 3, 6")

E[average^2] = 3/8,  E[g(Z1) g(Z2)] = 1/4,  m^2 = 1/4
E[average^2] = m^2 + sigma^2 / N for N = 2, 3, 6


## 8. Convolution (Theorem 18.17, Example 18.18, Figure 18.3, Exercise 18.7)

The density of a sum of independent variables is the convolution of their
densities, computed here on a grid as $\int f(z - y)g(y)\,dy$.

In [17]:
h = 1e-3
grid = torch.arange(-12.0, 12.0 + h / 2, h)             # symmetric grid, zero at the centre


def convolve(fv, gv):
    # (f * g)(z_j) = sum_i f(z_j - y_i) g(y_i) h, on the same grid ("same" convolution).
    out = torch.nn.functional.conv1d(fv.view(1, 1, -1), gv.flip(0).view(1, 1, -1),
                                     padding=(len(gv) - 1) // 2)
    return out.view(-1) * h


def at(values, z):
    return values[torch.argmin((grid - z).abs())].item()


uniform01 = ((grid >= 0) & (grid <= 1)).double()
tri = convolve(uniform01, uniform01)
for z in (0.3, 0.6, 1.0, 1.4, 1.9, 2.5):
    exact = max(0.0, 1 - abs(z - 1))
    print(f"uniform * uniform at z = {z}:  {at(tri, z):.4f}   triangle = {exact:.4f}")
    assert abs(at(tri, z) - exact) < 2e-3
assert abs(tri.sum().item() * h - 1) < 2e-3            # a probability density

expo = torch.where(grid > 0, torch.exp(-grid), torch.zeros_like(grid))
gam = convolve(expo, expo)
for z in (0.5, 1.0, 3.0):
    print(f"exponential * exponential at z = {z}:  {at(gam, z):.4f}   z e^-z = {z * math.exp(-z):.4f}")
    assert abs(at(gam, z) - z * math.exp(-z)) < 2e-3

phi = lambda s: torch.exp(-grid**2 / (2 * s**2)) / (s * math.sqrt(2 * math.pi))
s, t = 0.8, 1.5
r = math.sqrt(s**2 + t**2)
err = (convolve(phi(s), phi(t)) - phi(r)).abs().max().item()
print(f"max |phi_s * phi_t - phi_r| = {err:.2e}   with r^2 = s^2 + t^2 = {r**2:.2f}")
assert err < 1e-8

uniform * uniform at z = 0.3:  0.3010   triangle = 0.3000
uniform * uniform at z = 0.6:  0.6010   triangle = 0.6000
uniform * uniform at z = 1.0:  0.9990   triangle = 1.0000
uniform * uniform at z = 1.4:  0.5990   triangle = 0.6000
uniform * uniform at z = 1.9:  0.0990   triangle = 0.1000
uniform * uniform at z = 2.5:  0.0000   triangle = 0.0000


exponential * exponential at z = 0.5:  0.3039   z e^-z = 0.3033
exponential * exponential at z = 1.0:  0.3682   z e^-z = 0.3679
exponential * exponential at z = 3.0:  0.1494   z e^-z = 0.1494


max |phi_s * phi_t - phi_r| = 3.03e-15   with r^2 = s^2 + t^2 = 2.89


The first machine-learning box of §18.5. The empirical measure of a sample
has no density; its convolution with a Gaussian density is the kernel
density estimate $\frac1N\sum_i\varphi_\sigma(z - x_i)$. It is a probability
density, and its variance is the variance of the sample plus $\sigma^2$.

In [18]:
data = T([-2.0, -1.5, 0.3, 0.4, 0.5, 2.2])
sigma = 0.4
kde = torch.stack([torch.exp(-(grid - xi)**2 / (2 * sigma**2)) for xi in data]).mean(dim=0)
kde = kde / (sigma * math.sqrt(2 * math.pi))

mass = (kde * h).sum().item()
mean = (grid * kde * h).sum().item()
var = ((grid - mean)**2 * kde * h).sum().item()
print(f"mass = {mass:.8f}   mean = {mean:.6f} (sample mean {data.mean():.6f})")
print(f"variance = {var:.6f}   sample variance + sigma^2 = {data.var(unbiased=False) + sigma**2:.6f}")
assert abs(mass - 1) < 1e-8 and abs(mean - data.mean().item()) < 1e-8
assert abs(var - (data.var(unbiased=False).item() + sigma**2)) < 1e-7

mass = 1.00000000   mean = -0.016667 (sample mean -0.016667)
variance = 2.091389   sample variance + sigma^2 = 2.091389


## 9. Marginal, conditional and regression function (Propositions 18.19 and 18.20, Exercise 18.9)

The density $p(x, y) = x + y$ on the unit square. The marginal is
$x + \frac12$, the regression function is $m(x) = \frac{3x + 2}{6x + 3}$,
and for any predictor $h$,
$\mathbb E[(y - h(x))^2] = \mathbb E[(y - m(x))^2] + \mathbb E[(h(x) - m(x))^2]$.

In [19]:
n = 2000
g, hh = midpoints(0.0, 1.0, n)
X, Y = torch.meshgrid(g, g, indexing="ij")
p = X + Y
w = p * hh * hh                                       # the probability of each cell
print(f"total mass = {w.sum():.10f}")
assert abs(w.sum() - 1) < 1e-10

p_X = (p * hh).sum(dim=1)                             # integrate y out
assert torch.allclose(p_X, g + 0.5)
cond = p / p_X[:, None]                               # p(y | x)
assert torch.allclose((cond * hh).sum(dim=1), torch.ones(n))
m = (Y * cond * hh).sum(dim=1)                        # conditional mean
assert torch.allclose(m, (3 * g + 2) / (6 * g + 3), atol=1e-6)
print(f"m(0) ~ {m[0]:.4f} (2/3),  m(1) ~ {m[-1]:.4f} (5/9)")
assert not torch.allclose(p, (g[:, None] + 0.5) * (g[None, :] + 0.5))   # not independent

risk = lambda pred: ((Y - pred[:, None])**2 * w).sum().item()
best = risk(m)
exact_best = (12 - math.log(3)) / 144
print(f"risk of m             = {best:.8f}   (12 - ln 3)/144 = {exact_best:.8f}")
assert abs(best - exact_best) < 1e-6

predictors = {"constant E[y] = 7/12": torch.full((n,), 7 / 12), "h(x) = x": g.clone(),
              "h(x) = 1 - x": 1 - g}
for name, pred in predictors.items():
    gap = ((pred - m)**2 * p_X * hh).sum().item()     # int (h - m)^2 dP_X
    print(f"{name:22s} risk = {risk(pred):.8f}   risk(m) + gap = {best + gap:.8f}")
    assert abs(risk(pred) - (best + gap)) < 1e-9 and risk(pred) > best
assert abs(risk(predictors["constant E[y] = 7/12"]) - 11 / 144) < 1e-6

total mass = 1.0000000000
m(0) ~ 0.6666 (2/3),  m(1) ~ 0.5556 (5/9)
risk of m             = 0.07570406   (12 - ln 3)/144 = 0.07570408
constant E[y] = 7/12   risk = 0.07638887   risk(m) + gap = 0.07638887
h(x) = x               risk = 0.16666663   risk(m) + gap = 0.16666663


h(x) = 1 - x           risk = 0.16666663   risk(m) + gap = 0.16666663


The iterated form of the risk: averaging over $y$ given $x$ and then over
$x$ is the same as averaging over the pair. For the classification loss on
a finite set of labels, choosing the most probable label for each $x$
minimizes the risk (Exercise 18.9 (2)); all $3^4$ predictors are compared.

In [20]:
# x in {0, 1, 2, 3}, labels in {0, 1, 2}; a joint law given by a table of weights.
table = [[4, 1, 1], [1, 5, 2], [2, 2, 3], [3, 1, 6]]
total = sum(map(sum, table))
joint = {(x, y): Fraction(table[x][y], total) for x in range(4) for y in range(3)}
p_x = {x: sum(joint[(x, y)] for y in range(3)) for x in range(4)}


def error(pred):                                       # P(pred(x) != y)
    return sum(pr for (x, y), pr in joint.items() if pred[x] != y)


def error_iterated(pred):                              # E_x[ 1 - p(pred(x) | x) ]
    return sum(p_x[x] * (1 - joint[(x, pred[x])] / p_x[x]) for x in range(4))


bayes = tuple(max(range(3), key=lambda y: joint[(x, y)]) for x in range(4))
errors = {pred: error(pred) for pred in product(range(3), repeat=4)}
assert all(error(pred) == error_iterated(pred) for pred in errors)
best_pred = min(errors, key=errors.get)
print(f"most probable labels: {bayes}   error = {errors[bayes]}")
print(f"best of the {len(errors)} predictors: {best_pred}   error = {errors[best_pred]}")
assert best_pred == bayes and errors[bayes] == 1 - Fraction(4 + 5 + 3 + 6, total)

most probable labels: (0, 1, 2, 2)   error = 13/31
best of the 81 predictors: (0, 1, 2, 2)   error = 13/31


## Summary

* On finite sets the product measure is computed point by point or by
  slices in either direction, with the same result.
* $\int f\,d\mu = \int_0^\infty\mu\{f > t\}\,dt$; the mean of a sample is the
  integral of its tail function.
* Tonelli: $\sum_{n\geq2}(\zeta(n) - 1) = 1$,
  $\int_0^1\int_y^1e^{x^2}dx\,dy = (e - 1)/2$,
  $\int_0^\infty(e^{-ax} - e^{-bx})/x\,dx = \ln(b/a)$,
  $\int_0^\infty e^{-x^2}dx = \sqrt\pi/2$.
* The exchange fails for the $\pm1$ array ($0$ and $1$) and for
  $(x^2 - y^2)/(x^2 + y^2)^2$ ($\pi/4$ and $-\pi/4$), whose absolute value
  has an infinite integral.
* The simplex has volume $1/n!$.
* For independent variables expectations of products factor and variances
  add; an average of $N$ samples has variance $\sigma^2/N$, in dimension $1$
  as in dimension $50$. The square of an average is a biased estimate of
  $m^2$.
* Uniform $*$ uniform is a triangle, exponential $*$ exponential is
  $ze^{-z}$, Gaussian variances add, and a kernel density estimate is the
  empirical measure convolved with the kernel.
* For $p(x, y) = x + y$ the regression function is $(3x + 2)/(6x + 3)$ and
  the squared risk splits into the risk of $m$ plus the distance to $m$.